In [13]:
from pathlib import Path
import pandas as pd
import shutil

# =========================
# 1. 配置路径
# =========================

# case_00xx 这一层路径
CASE_DIR = Path(r"C:\Users\wxy\Desktop\move_with_data_images\fetal_hydro_dataset_3_report_合并 copy\raw\case_0009")

# 标注结果文件名
CSV_NAME = "标注结果.csv"

# 图像所在文件夹名
IMAGE_FOLDER_NAME = "with_data_images"

# 是否真正执行文件移动/复制
# 第一次建议设为 False，只打印将要执行的操作
# 确认无误后再改为 True
DO_ACTION = True


# =========================
# 2. 读取 CSV 的函数
# =========================

def read_csv_safely(csv_path: Path):
    """
    自动尝试常见编码读取 CSV。
    """
    for enc in ["utf-8-sig", "utf-8", "gbk"]:
        try:
            return pd.read_csv(csv_path, encoding=enc)
        except UnicodeDecodeError:
            continue

    raise UnicodeDecodeError(
        "无法读取 CSV，请检查文件编码",
        b"",
        0,
        1,
        "unknown encoding"
    )


# =========================
# 3. 从 CSV 中提取图像文件名
# =========================

def get_image_name(file_value):
    """
    CSV 的 文件名 列可能是：
    1. 0_0.jpg
    2. with_data_images/0_0.jpg
    3. case_0004/1prenatal/202411270182/with_data_images/0_0.jpg

    这里统一只取最后一级文件名。
    """
    return Path(str(file_value).replace("\\", "/")).name


# =========================
# 4. 处理单次检查
# =========================

def process_one_exam(exam_dir: Path):
    """
    处理一个检查文件夹，例如：
    case_0004/1prenatal/202411270182
    """

    csv_path = exam_dir / CSV_NAME
    image_dir = exam_dir / IMAGE_FOLDER_NAME

    if not csv_path.exists():
        return

    if not image_dir.exists():
        print(f"[跳过] 找到 CSV，但不存在图像文件夹: {image_dir}")
        return

    left_img_dir = exam_dir / "left" / "images"
    right_img_dir = exam_dir / "right" / "images"

    if DO_ACTION:
        left_img_dir.mkdir(parents=True, exist_ok=True)
        right_img_dir.mkdir(parents=True, exist_ok=True)

    df = read_csv_safely(csv_path)

    # 检查列名
    required_cols = {"文件名", "评分"}
    if not required_cols.issubset(set(df.columns)):
        print(f"[跳过] CSV 缺少必要列: {csv_path}")
        print(f"当前列名: {list(df.columns)}")
        return

    print(f"\n[处理检查] {exam_dir}")
    print(f"[CSV] {csv_path}")

    for idx, row in df.iterrows():
        image_name = get_image_name(row["文件名"])
        score = str(row["评分"]).strip().lower()

        src_img = image_dir / image_name

        if not src_img.exists():
            print(f"  [未找到图像] {src_img}")
            continue

        dst_left = left_img_dir / image_name
        dst_right = right_img_dir / image_name

        if score == "left":
            print(f"  [left] 移动: {src_img} -> {dst_left}")
            if DO_ACTION:
                shutil.move(str(src_img), str(dst_left))

        elif score == "right":
            print(f"  [right] 移动: {src_img} -> {dst_right}")
            if DO_ACTION:
                shutil.move(str(src_img), str(dst_right))

        elif score == "both":
            print(f"  [both] 复制到左右两侧，并删除原图: {src_img}")
            if DO_ACTION:
                shutil.copy2(str(src_img), str(dst_left))
                shutil.copy2(str(src_img), str(dst_right))
                src_img.unlink()

        elif score == "neither":
            print(f"  [neither] 不处理: {src_img}")

        else:
            print(f"  [未知评分] 第 {idx + 1} 行，图像: {image_name}，评分: {score}")


# =========================
# 5. 遍历 case 下所有检查
# =========================

def process_case(case_dir: Path):
    """
    自动遍历 case_0004 下所有阶段和检查：
    case_0004/
        1prenatal/
            202411270182/
            202412231062/
        2postnatal/
            ...
        3postoperative/
            ...
    """

    if not case_dir.exists():
        raise FileNotFoundError(f"case 路径不存在: {case_dir}")

    for phase_dir in case_dir.iterdir():
        if not phase_dir.is_dir():
            continue

        for exam_dir in phase_dir.iterdir():
            if not exam_dir.is_dir():
                continue

            process_one_exam(exam_dir)


# =========================
# 6. 开始执行
# =========================

process_case(CASE_DIR)


[处理检查] C:\Users\wxy\Desktop\move_with_data_images\fetal_hydro_dataset_3_report_合并 copy\raw\case_0009\1prenatal\202412010645
[CSV] C:\Users\wxy\Desktop\move_with_data_images\fetal_hydro_dataset_3_report_合并 copy\raw\case_0009\1prenatal\202412010645\标注结果.csv
  [both] 复制到左右两侧，并删除原图: C:\Users\wxy\Desktop\move_with_data_images\fetal_hydro_dataset_3_report_合并 copy\raw\case_0009\1prenatal\202412010645\with_data_images\8_8.jpg

[处理检查] C:\Users\wxy\Desktop\move_with_data_images\fetal_hydro_dataset_3_report_合并 copy\raw\case_0009\1prenatal\202502110417
[CSV] C:\Users\wxy\Desktop\move_with_data_images\fetal_hydro_dataset_3_report_合并 copy\raw\case_0009\1prenatal\202502110417\标注结果.csv
  [left] 移动: C:\Users\wxy\Desktop\move_with_data_images\fetal_hydro_dataset_3_report_合并 copy\raw\case_0009\1prenatal\202502110417\with_data_images\46_46.jpg -> C:\Users\wxy\Desktop\move_with_data_images\fetal_hydro_dataset_3_report_合并 copy\raw\case_0009\1prenatal\202502110417\left\images\46_46.jpg
  [both] 复制到左右两侧，并删除原